# NLP with Disaster Tweets — baseline reproduzível

Este notebook compara um baseline majoritário com TF-IDF + Logistic Regression para classificar tweets em duas classes. O foco é demonstrar um pipeline rastreável: separação de dados sem vazamento, métricas na validação, análise de erros e geração de arquivo de submissão.

**Antes de executar:** aceite as regras da competição Kaggle Natural Language Processing with Disaster Tweets e adicione o dataset ao notebook. Não use os rótulos de teste na validação.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

SEED = 42
INPUT_DIR = Path('/kaggle/input/nlp-getting-started')
TRAIN_PATH = INPUT_DIR / 'train.csv'
TEST_PATH = INPUT_DIR / 'test.csv'

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError(
        'Dataset não encontrado. Adicione a competição ao notebook e confira INPUT_DIR.'
    )

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
print('Train:', train.shape, '| Test:', test.shape)
train.head()


## 1. Inspeção inicial
Verifique distribuição de classes, campos, texto vazio e duplicatas antes de modelar.


In [ ]:
display(train.head())
display(train.isna().sum().to_frame('missing_values'))
display(train['target'].value_counts(normalize=True).rename('proportion'))
print('Textos duplicados:', train['text'].duplicated().sum())
print('Classes:', sorted(train['target'].dropna().unique().tolist()))

train['text'] = train['text'].fillna('').astype(str)
test['text'] = test['text'].fillna('').astype(str)


## 2. Divisão treino/validação
A divisão é estratificada e fixa para tornar a comparação entre modelos mais consistente. O conjunto oficial de teste permanece separado para a submissão final.


In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    train['text'], train['target'], test_size=0.20,
    random_state=SEED, stratify=train['target']
)
print(f'Treino: {len(X_train)} | Validação: {len(X_valid)}')


## 3. Baseline e modelo TF-IDF
O DummyClassifier fornece uma referência simples. Em seguida, o pipeline converte texto em n-gramas TF-IDF e treina uma regressão logística. O vocabulário é ajustado somente com X_train para evitar vazamento da validação.


In [ ]:
models = {
    'Dummy (most_frequent)': DummyClassifier(strategy='most_frequent'),
    'TF-IDF + Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(
            lowercase=True, strip_accents='unicode',
            ngram_range=(1, 2), min_df=2,
            max_features=50000, sublinear_tf=True
        )),
        ('classifier', LogisticRegression(max_iter=1000, random_state=SEED))
    ])
}

rows = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)
    predictions[name] = pred
    rows.append({
        'model': name,
        'precision': precision_score(y_valid, pred, zero_division=0),
        'recall': recall_score(y_valid, pred, zero_division=0),
        'f1': f1_score(y_valid, pred, zero_division=0),
    })

metrics = pd.DataFrame(rows).sort_values('f1', ascending=False)
display(metrics)


## 4. Análise do modelo selecionado
Use o F1 da validação para selecionar o modelo desta experiência. A matriz de confusão e os erros ajudam a entender que tipos de textos são classificados incorretamente.


In [ ]:
best_name = metrics.iloc[0]['model']
best_model = models[best_name]
valid_pred = predictions[best_name]
print('Modelo selecionado na validação:', best_name)
print(classification_report(y_valid, valid_pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_valid, valid_pred)
plt.title(f'Matriz de confusão — {best_name}')
plt.tight_layout()
plt.show()

error_analysis = pd.DataFrame({
    'text': X_valid.reset_index(drop=True),
    'actual': y_valid.reset_index(drop=True),
    'predicted': pd.Series(valid_pred).reset_index(drop=True),
})
errors = error_analysis[error_analysis['actual'] != error_analysis['predicted']]
print('Quantidade de erros:', len(errors), 'de', len(error_analysis))
display(errors.head(15))


## 5. Treinamento final e arquivo de submissão
Depois de concluir a análise de validação, treine a configuração selecionada em todos os exemplos rotulados e gere o formato esperado pela competição.

Observação: selecionar entre modelos com base em uma única divisão pode produzir estimativas instáveis. Para um estudo mais robusto, experimente validação cruzada e registre o protocolo utilizado.


In [ ]:
final_model = models[best_name]
final_model.fit(train['text'], train['target'])
test_predictions = final_model.predict(test['text']).astype(int)

submission = pd.DataFrame({
    'id': test['id'],
    'target': test_predictions,
})
assert list(submission.columns) == ['id', 'target']
assert len(submission) == len(test)
assert submission['target'].isin([0, 1]).all()

output_path = Path('/kaggle/working/submission.csv')
submission.to_csv(output_path, index=False)
print('Arquivo salvo em:', output_path)
display(submission.head())


## 6. Registro experimental
Após executar, registre aqui a versão das bibliotecas, o F1 de validação obtido, a configuração vencedora e as limitações observadas. Não confunda F1 local de validação com pontuação pública/privada do leaderboard: esta última só existe após submeter o arquivo na competição.


In [ ]:
import sklearn
print('scikit-learn:', sklearn.__version__)
print('Seed:', SEED)
print('Métricas de validação:')
display(metrics)
